In [0]:
!pip install transformers
!pip install torch
!pip install torchvision

In [0]:
%restart_python

In [0]:
%sql
CREATE TABLE workspace.default.product_reviews (
  review_id STRING,
  customer_name STRING,
  rating INT,
  review_text STRING,
  review_date DATE
)
USING DELTA;

In [0]:
%sql
INSERT INTO workspace.default.product_reviews VALUES
-- Review 1
('r001', 'John D', 5,
'This product exceeded my expectations. 

The build quality is top-notch and feels very premium in hand. I was particularly impressed by the packaging – it was neat, secure, and had zero damage.

I have been using it for over a week now and there have been no issues whatsoever. Highly recommended!',
'2024-07-01'),

-- Review 2
('r002', 'Sara M', 3,
'The product is decent for the price.

However, I was disappointed by the packaging. The box was partially torn and one corner of the item had scratches.

Overall, it works fine, but I expected better presentation.',
'2024-07-02')

In [0]:
!pip install transformers
!pip install torch
!pip install torchvision

In [0]:
%pip install transformers torch torchvision

In [0]:
dbutils.library.restartPython()

In [0]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

tokenizer = T5Tokenizer.from_pretrained("t5-small")
model = T5ForConditionalGeneration.from_pretrained("t5-small")

text = "summarize: Databricks makes data teams more productive with a unified analytics platform."
inputs = tokenizer(text, return_tensors="pt", max_length=512, truncation=True)
outputs = model.generate(inputs.input_ids, min_length=20, max_length=50)
summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(summary)

In [0]:
import mlflow
from mlflow.models import infer_signature

# It is valuable to log a "signature" with the model, telling MLflow the input and output schema
output = summary
signature = infer_signature(text, output)
print(f"Signature:\n{signature}\n")

# Set experiment path (visible in sidebar under Machine Learning -> Experiments)
experiment_name = "/Users/santi1947@gmail.com/GenAI/poc/summarizer_experiment"
mlflow.set_experiment(experiment_name)

model_artifact_path = "summarizer"  # Name of folder containing serialized model
# The following text is commented out as per instructions
# from mlflow.transformers import generate_signature_output
# output = generate_signature_output(summarizer, text)

In [0]:
with mlflow.start_run():
    # LOG PARAMS
    mlflow.log_params({
        "hf_model_name": "t5-small"
    })
    mlflow.transformers.log_model(
        transformers_model=summarizer,
        artifact_path=model_artifact_path,
        input_example=text,
        signature=signature
    )